## **03. Modelisation & Comparaison**

In [1]:
# Import 
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# constant 
output_data_path = '../output/data'
output_models_path = '../output/models'
output_images_path = '../output/images'

# Chargement du dataset nettoyé
df = pd.read_csv(f"{output_data_path}/df_preprocessed.csv")
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Day,DayOfWeek
0,6,2011-02-18,1572117.54,0,59.61,3.045,214.777523,6.858,2011,2,18,4
1,13,2011-03-25,1807545.43,0,42.38,3.435,128.616064,7.470,2011,3,25,4
2,6,2010-05-28,1644470.66,0,78.89,2.759,212.412888,7.092,2010,5,28,4
3,4,2010-05-28,1857533.70,0,62.25,2.756,126.160226,7.896,2010,5,28,4
4,15,2011-06-03,695396.19,0,69.80,4.069,134.855161,7.658,2011,6,3,4


### **Encodage et Train/Test Split**

In [21]:
# Encodage 
cat_col = ['Store']
# Exclure Date : lue en texte depuis le CSV, incompatible avec sklearn ; Year/Month/Day/DayOfWeek suffisent
num_cols = [c for c in df.columns if c not in cat_col + ['Weekly_Sales', 'Date']]

print(f"Variable catégorielle : {cat_col}")
print(f"Variables numériques  : {num_cols}")
print(f"Target : Weekly_Sales")

y = df['Weekly_Sales']
X_raw = df.drop(columns=['Weekly_Sales'])

# ColumnTransformer : OneHotEncoder sur Store, passthrough sur le reste
ct = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='error'), cat_col),
        ('num', 'passthrough', num_cols)
    ],
    remainder='drop'
)
ct.fit(X_raw)

# Récupérer les noms de colonnes
ohe_feature_names = ct.named_transformers_['cat'].get_feature_names_out(cat_col).tolist()
all_feature_names = ohe_feature_names + num_cols

# Transformer
X_transformed = ct.transform(X_raw)
X = pd.DataFrame(X_transformed, columns=all_feature_names, index=X_raw.index)
y = y.reset_index(drop=True)
X = X.reset_index(drop=True)

# Sauvegarder le ColumnTransformer
joblib.dump(ct, f"{output_models_path}/column_transformer.pkl")
print(f"ColumnTransformer sauvegardé : {output_models_path}/column_transformer.pkl")

print(X.head(), y.head())

Variable catégorielle : ['Store']
Variables numériques  : ['Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Day', 'DayOfWeek']
Target : Weekly_Sales
ColumnTransformer sauvegardé : ../output/models/column_transformer.pkl
   Store_2  Store_3  Store_4  Store_5  Store_6  Store_7  Store_8  Store_9  \
0      0.0      0.0      0.0      0.0      1.0      0.0      0.0      0.0   
1      0.0      0.0      0.0      0.0      0.0      0.0      0.0      0.0   
2      0.0      0.0      0.0      0.0      1.0      0.0      0.0      0.0   
3      0.0      0.0      1.0      0.0      0.0      0.0      0.0      0.0   
4      0.0      0.0      0.0      0.0      0.0      0.0      0.0      0.0   

   Store_10  Store_11  ...  Store_20  Holiday_Flag  Temperature  Fuel_Price  \
0       0.0       0.0  ...       0.0           0.0        59.61       3.045   
1       0.0       0.0  ...       0.0           0.0        42.38       3.435   
2       0.0       0.0  ...       0.0       

In [3]:
# SPLIT TRAIN / TEST 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Total obs: {len(X)} ")
print(f"Train dataset (80%) : {len(X_train)} ")
print(f"Test dataset (20%) : {len(X_test)} ")


Total obs: 113 
Train dataset (80%) : 90 
Test dataset (20%) : 23 


### **Entrainement Models**

#### **Model 1. Baseline: Regression Linéaire**

In [4]:
lr = LinearRegression()
lr.fit(X_train, y_train)
 
y_train_pred_lr = lr.predict(X_train)
y_test_pred_lr = lr.predict(X_test)
 
rmse_train_lr = np.sqrt(mean_squared_error(y_train, y_train_pred_lr))
rmse_test_lr = np.sqrt(mean_squared_error(y_test, y_test_pred_lr))
mae_train_lr = mean_absolute_error(y_train, y_train_pred_lr)
mae_test_lr = mean_absolute_error(y_test, y_test_pred_lr)
r2_train_lr = r2_score(y_train, y_train_pred_lr)
r2_test_lr = r2_score(y_test, y_test_pred_lr)
 
print(f"Modèle entraîné (intercept = {lr.intercept_:,.2f})")
print(f"\n{'Métrique':<10} {'Train':>15} {'Test':>15} {'Gap':>12}")

print(f"\n{'RMSE':<10} {rmse_train_lr:>15,.2f} {rmse_test_lr:>15,.2f} {rmse_test_lr-rmse_train_lr:>+12,.2f}")
print(f"\n{'MAE':<10} {mae_train_lr:>15,.2f} {mae_test_lr:>15,.2f} {mae_test_lr-mae_train_lr:>+12,.2f}")
print(f"\n{'R_Carré':<10} {r2_train_lr:>15.4f} {r2_test_lr:>15.4f} {r2_test_lr-r2_train_lr:>+12.4f}")

Modèle entraîné (intercept = 57,998,579.56)

Métrique             Train            Test          Gap

RMSE            108,285.01      151,200.29   +42,915.28

MAE              80,405.41      106,304.11   +25,898.70

R_Carré             0.9755          0.9327      -0.0429


#### **Regularisation avec Ridge**

In [5]:
ridge_params = {'alpha': [0.01, 0.1, 1, 5, 10, 50, 100, 500, 1000, 5000, 10000]}
ridge_cv = GridSearchCV(
    Ridge(), ridge_params, cv=5, scoring='r2', return_train_score=True, n_jobs=-1
)
ridge_cv.fit(X_train, y_train)
 
best_ridge = ridge_cv.best_estimator_
print(f"GridSearchCV terminé (5-fold CV)")
print(f"Meilleur alpha  : {ridge_cv.best_params_['alpha']}")
print(f"Meilleur R carré CV  : {ridge_cv.best_score_:.4f}") 

y_train_pred_ridge = best_ridge.predict(X_train)
y_test_pred_ridge = best_ridge.predict(X_test)
 
rmse_train_ridge = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
rmse_test_ridge = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge))
mae_train_ridge = mean_absolute_error(y_train, y_train_pred_ridge)
mae_test_ridge = mean_absolute_error(y_test, y_test_pred_ridge)
r2_train_ridge = r2_score(y_train, y_train_pred_ridge)
r2_test_ridge = r2_score(y_test, y_test_pred_ridge)

print(f"\n{'Métrique':<10} {'Train':>15} {'Test':>15} {'Gap':>12}")
print(f"{'RMSE':<10} {rmse_train_ridge:>15,.2f} {rmse_test_ridge:>15,.2f} {rmse_test_ridge-rmse_train_ridge:>+12,.2f}")
print(f"{'MAE':<10} {mae_train_ridge:>15,.2f} {mae_test_ridge:>15,.2f} {mae_test_ridge-mae_train_ridge:>+12,.2f}")
print(f"{'R_Carré':<10} {r2_train_ridge:>15.4f} {r2_test_ridge:>15.4f} {r2_test_ridge-r2_train_ridge:>+12.4f}")


GridSearchCV terminé (5-fold CV)
Meilleur alpha  : 0.1
Meilleur R carré CV  : 0.9079

Métrique             Train            Test          Gap
RMSE            111,005.45      163,488.97   +52,483.52
MAE              83,539.99      110,861.61   +27,321.62
R_Carré             0.9743          0.9213      -0.0530


#### **Regularisation avec Lasso**

In [6]:
lasso_params = {'alpha': [1, 10, 50, 100, 500, 1000, 2000, 5000, 10000, 20000, 50000]}
lasso_cv = GridSearchCV(
    Lasso(max_iter=50000), lasso_params, cv=5, scoring='r2', return_train_score=True, n_jobs=-1
)
lasso_cv.fit(X_train, y_train)
 
best_lasso = lasso_cv.best_estimator_
print(f"GridSearchCV terminé (5-fold CV)")
print(f"Meilleur alpha  : {lasso_cv.best_params_['alpha']}")
print(f"Meilleur R carré CV  : {lasso_cv.best_score_:.4f}")

y_train_pred_lasso = best_lasso.predict(X_train)
y_test_pred_lasso = best_lasso.predict(X_test)
 
rmse_train_lasso = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso))
rmse_test_lasso = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso))
mae_train_lasso = mean_absolute_error(y_train, y_train_pred_lasso)
mae_test_lasso = mean_absolute_error(y_test, y_test_pred_lasso)
r2_train_lasso = r2_score(y_train, y_train_pred_lasso)
r2_test_lasso = r2_score(y_test, y_test_pred_lasso)
 
print(f"\n{'Métrique':<10} {'Train':>15} {'Test':>15} {'Gap':>12}")
print(f"{'RMSE':<10} {rmse_train_lasso:>15,.2f} {rmse_test_lasso:>15,.2f} {rmse_test_lasso-rmse_train_lasso:>+12,.2f}")
print(f"{'MAE':<10} {mae_train_lasso:>15,.2f} {mae_test_lasso:>15,.2f} {mae_test_lasso-mae_train_lasso:>+12,.2f}")
print(f"{'R_Carré':<10} {r2_train_lasso:>15.4f} {r2_test_lasso:>15.4f} {r2_test_lasso-r2_train_lasso:>+12.4f}")

GridSearchCV terminé (5-fold CV)
Meilleur alpha  : 1000
Meilleur R carré CV  : 0.9093

Métrique             Train            Test          Gap
RMSE            111,792.47      166,445.78   +54,653.31
MAE              85,012.76      118,522.03   +33,509.27
R_Carré             0.9739          0.9184      -0.0555


In [ ]:
# Variables éliminées par Lasso
lasso_coefs = pd.Series(best_lasso.coef_, index=X_train.columns)
eliminated = lasso_coefs[lasso_coefs == 0].index.tolist()
kept = lasso_coefs[lasso_coefs != 0].index.tolist()

print(f"\nSélection de variables par Lasso")
print(f"Variables conservées  : {len(kept)}/{len(lasso_coefs)}")
print(f"Variables éliminées   : {len(eliminated)}")

if eliminated:
    print(f"\nÉliminées : {eliminated}")


Sélection de variables par Lasso
   Variables conservées  : 24/27
   Variables éliminées   : 3
Éliminées : ['Store_19', 'Holiday_Flag', 'DayOfWeek']


### **COMPARAISON DES 3 MODÈLES**

In [8]:
comparison = pd.DataFrame({
    'Modèle': ['LinearRegression', f'Ridge (o={ridge_cv.best_params_["alpha"]})',
               f'Lasso (o={lasso_cv.best_params_["alpha"]})'],
    'RMSE_train': [rmse_train_lr, rmse_train_ridge, rmse_train_lasso],
    'RMSE_test': [rmse_test_lr, rmse_test_ridge, rmse_test_lasso],
    'MAE_train': [mae_train_lr, mae_train_ridge, mae_train_lasso],
    'MAE_test': [mae_test_lr, mae_test_ridge, mae_test_lasso],
    'R2_train': [r2_train_lr, r2_train_ridge, r2_train_lasso],
    'R2_test': [r2_test_lr, r2_test_ridge, r2_test_lasso],
})
comparison['Gap_R2'] = comparison['R2_train'] - comparison['R2_test']
comparison['RMSE_pct'] = (comparison['RMSE_test'] / y_test.mean() * 100).round(1)
 
print("Tableau comparatif")
print(f"{'Modèle':<25} {'RMSE test':>12} {'MAE test':>12} {'R_Carré test':>10} {'Gap R_Carré':>10} {'RMSE %':>8}")
for _, row in comparison.iterrows():
    print(f"{row['Modèle']:<25} {row['RMSE_test']:>12,.0f} {row['MAE_test']:>12,.0f} {row['R2_test']:>10.4f} {row['Gap_R2']:>10.4f} {row['RMSE_pct']:>7.1f}%")

Tableau comparatif
Modèle                       RMSE test     MAE test R_Carré test Gap R_Carré   RMSE %
LinearRegression               151,200      106,304     0.9327     0.0429    11.6%
Ridge (o=0.1)                  163,489      110,862     0.9213     0.0530    12.5%
Lasso (o=1000)                 166,446      118,522     0.9184     0.0555    12.7%


In [9]:
# Déterminer le meilleur modèle
best_idx = comparison['R2_test'].idxmax()
best_model_name = comparison.loc[best_idx, 'Modèle']
best_r2 = comparison.loc[best_idx, 'R2_test']
print(f"\nMeilleur modèle : {best_model_name} (R_Carré test = {best_r2:.4f})")


Meilleur modèle : LinearRegression (R_Carré test = 0.9327)


In [10]:
# Overfitting check
print(f"\nVérification overfitting")
for _, row in comparison.iterrows():
    gap = row['Gap_R2']
    status = "OK" if gap < 0.05 else "Léger" if gap < 0.10 else "Overfitting"
    print(f"   {row['Modèle']:<25} : Gap R_Carré = {gap:.4f}  {status}")


Vérification overfitting
   LinearRegression          : Gap R_Carré = 0.0429  OK
   Ridge (o=0.1)             : Gap R_Carré = 0.0530  Léger
   Lasso (o=1000)            : Gap R_Carré = 0.0555  Léger


In [11]:
# ANALYSE DES COEFFICIENTS (3 modèles)

coef_compare = pd.DataFrame({
    'Feature': X_train.columns,
    'LinearReg': lr.coef_,
    'Ridge': best_ridge.coef_,
    'Lasso': best_lasso.coef_
}).set_index('Feature')
 
# Top features hors Store dummies
coef_no_store = coef_compare[~coef_compare.index.str.startswith('Store_')]

print("Coefficients des variables principales")
print(f"{'Feature':<15} {'LinearReg':>14} {'Ridge':>14} {'Lasso':>14}")
for feat in coef_no_store.index:
    print(f"{feat:<15} {coef_no_store.loc[feat,'LinearReg']:>14,.2f} {coef_no_store.loc[feat,'Ridge']:>14,.2f} {coef_no_store.loc[feat,'Lasso']:>14,.2f}")
 
# Effet de la régularisation
print(f"\nEffet de la régularisation")
lr_norm = np.sqrt((lr.coef_**2).sum())
ridge_norm = np.sqrt((best_ridge.coef_**2).sum())
lasso_norm = np.sqrt((best_lasso.coef_**2).sum())

print(f"Norme L2 des coefficients :")
print(f"LinearRegression : {lr_norm:>14,.2f}")
print(f"Ridge : {ridge_norm:>14,.2f}  (réduction : {(1-ridge_norm/lr_norm)*100:.1f}%)")
print(f"Lasso : {lasso_norm:>14,.2f}  (réduction : {(1-lasso_norm/lr_norm)*100:.1f}%)")

Coefficients des variables principales
Feature              LinearReg          Ridge          Lasso
Holiday_Flag        -52,233.91     -29,674.09          -0.00
Temperature          -1,833.87      -1,862.01      -1,841.31
Fuel_Price          -56,744.06     -56,680.19     -42,127.92
CPI                   2,439.11       1,918.13       1,683.68
Unemployment        -37,494.00      -7,211.80     -14,315.62
Year                -28,053.13     -12,980.80     -22,460.39
Month                18,473.76      19,069.33      18,553.52
Day                  -5,625.17      -5,655.97      -5,490.19
DayOfWeek                 0.00           0.00           0.00

Effet de la régularisation
Norme L2 des coefficients :
LinearRegression :   3,107,554.33
Ridge :   2,912,906.80  (réduction : 6.3%)
Lasso :   2,948,914.32  (réduction : 5.1%)


In [12]:
# Analyse des résidus

models_pred = {
    'LinearReg': y_test_pred_lr,
    'Ridge': y_test_pred_ridge,
    'Lasso': y_test_pred_lasso
}
 
print(f"{'Modèle':<15} {'Moy résidu':>14} {'Std résidu':>14} {'Min':>14} {'Max':>14}")
for name, pred in models_pred.items():
    residuals = y_test.values - pred
    print(f"{name:<15} {residuals.mean():>14,.2f} {residuals.std():>14,.2f} {residuals.min():>14,.2f} {residuals.max():>14,.2f}")


Modèle              Moy résidu     Std résidu            Min            Max
LinearReg            32,435.59     147,680.27    -208,621.96     507,278.33
Ridge                31,168.56     160,490.38    -215,801.35     552,260.85
Lasso                21,206.13     165,089.36    -249,813.68     556,029.02


In [13]:
# VISUALISATIONS
model_colors = {'LinearReg': '#3498db', 'Ridge': '#e67e22', 'Lasso': '#27ae60'}

# Fig 1 : Comparaison des métriques (barplot groupé) -
fig1 = make_subplots(rows=1, cols=3, subplot_titles=("RMSE", "MAE", "R_Carré")) 
 
for i, (metric, train_vals, test_vals) in enumerate([
    ("RMSE", [rmse_train_lr, rmse_train_ridge, rmse_train_lasso],
             [rmse_test_lr, rmse_test_ridge, rmse_test_lasso]),
    ("MAE", [mae_train_lr, mae_train_ridge, mae_train_lasso],
            [mae_test_lr, mae_test_ridge, mae_test_lasso]),
    ("R_Carré", [r2_train_lr, r2_train_ridge, r2_train_lasso],
           [r2_test_lr, r2_test_ridge, r2_test_lasso]),
]):
    names = ['LR', 'Ridge', 'Lasso']
    fig1.add_trace(go.Bar(x=names, y=train_vals, name='Train' if i == 0 else None,
                           marker_color='#3498db', opacity=0.6,
                           showlegend=(i == 0), legendgroup='train',
                           text=[f"{v:,.2f}" for v in train_vals],
                           textposition='outside', textfont=dict(size=9)), row=1, col=i+1)
    fig1.add_trace(go.Bar(x=names, y=test_vals, name='Test' if i == 0 else None,
                           marker_color='#e74c3c', opacity=0.8,
                           showlegend=(i == 0), legendgroup='test',
                           text=[f"{v:,.2f}" for v in test_vals],
                           textposition='outside', textfont=dict(size=9)), row=1, col=i+1)
 
fig1.update_layout(title="Fig.1 - Comparaison des métriques : Train vs Test",
                   height=450, width=1050, template='plotly_white', barmode='group')
fig1.write_image(f"{output_images_path}/f1_metrics_comparison.png", scale=2)
fig1.show()

In [14]:
# Fig 2 : Actual vs Predicted (3 modèles) 
fig2 = make_subplots(rows=1, cols=3,
                     subplot_titles=("LinearRegression", "Ridge", "Lasso"))
 
all_vals = np.concatenate([y_train, y_test])
min_val, max_val = all_vals.min(), all_vals.max()
 
for i, (name, pred, color) in enumerate([
    ('LinearReg', y_test_pred_lr, '#3498db'),
    ('Ridge', y_test_pred_ridge, '#e67e22'),
    ('Lasso', y_test_pred_lasso, '#27ae60')
]):
    fig2.add_trace(go.Scatter(x=y_test, y=pred, mode='markers',
                               marker=dict(color=color, size=8, opacity=0.7),
                               showlegend=False), row=1, col=i+1)
    fig2.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
                               mode='lines', line=dict(color='red', dash='dash', width=2),
                               showlegend=False), row=1, col=i+1)
    fig2.update_xaxes(title_text="Réel", row=1, col=i+1)
    fig2.update_yaxes(title_text="Prédit", row=1, col=i+1)
 
fig2.update_layout(title="Fig.2 - Actual vs Predicted (test set)",
                   height=400, width=1100, template='plotly_white')
fig2.write_image(f"{output_images_path}/f2_actual_vs_predicted.png", scale=2)
fig2.show()

In [15]:
 
# Fig 3 : Résidus des 3 modèles 
fig3 = make_subplots(rows=1, cols=3,
                     subplot_titles=("LinearRegression", "Ridge", "Lasso"))
 
for i, (name, pred, color) in enumerate([
    ('LinearReg', y_test_pred_lr, '#3498db'),
    ('Ridge', y_test_pred_ridge, '#e67e22'),
    ('Lasso', y_test_pred_lasso, '#27ae60')
]):
    residuals = y_test.values - pred
    fig3.add_trace(go.Scatter(x=pred, y=residuals, mode='markers',
                               marker=dict(color=color, size=7, opacity=0.7),
                               showlegend=False), row=1, col=i+1)
    fig3.add_hline(y=0, line_color="red", line_dash="dash", row=1, col=i+1)
    fig3.update_xaxes(title_text="Prédictions", row=1, col=i+1)
    fig3.update_yaxes(title_text="Résidus", row=1, col=i+1)
 
fig3.update_layout(title="Fig.3 - Résidus (test set)",
                   height=400, width=1100, template='plotly_white')
fig3.write_image(f"{output_images_path}/f3_residuals.png", scale=2)
fig3.show()

In [16]:
# Fig 4 : GridSearchCV - R_Carré vs Alpha 
fig4 = make_subplots(rows=1, cols=2, subplot_titles=("Ridge", "Lasso"))
 
# Ridge
cv_results_ridge = pd.DataFrame(ridge_cv.cv_results_)
ridge_alphas = cv_results_ridge['param_alpha'].astype(float)
fig4.add_trace(go.Scatter(x=ridge_alphas, y=cv_results_ridge['mean_test_score'],
                           mode='lines+markers', marker=dict(color='#e67e22', size=7),
                           line=dict(color='#e67e22'), name='Ridge CV',
                           showlegend=False), row=1, col=1)
fig4.add_vline(x=ridge_cv.best_params_['alpha'], line_dash="dash", line_color="red",
               row=1, col=1, annotation_text=f"best α={ridge_cv.best_params_['alpha']}")
fig4.update_xaxes(type="log", title_text="Alpha", row=1, col=1)
fig4.update_yaxes(title_text="R_Carré (CV)", row=1, col=1)
 
# Lasso
cv_results_lasso = pd.DataFrame(lasso_cv.cv_results_)
lasso_alphas = cv_results_lasso['param_alpha'].astype(float)
fig4.add_trace(go.Scatter(x=lasso_alphas, y=cv_results_lasso['mean_test_score'],
                           mode='lines+markers', marker=dict(color='#27ae60', size=7),
                           line=dict(color='#27ae60'), name='Lasso CV',
                           showlegend=False), row=1, col=2)
fig4.add_vline(x=lasso_cv.best_params_['alpha'], line_dash="dash", line_color="red",
               row=1, col=2, annotation_text=f"best α={lasso_cv.best_params_['alpha']}")
fig4.update_xaxes(type="log", title_text="Alpha", row=1, col=2)
fig4.update_yaxes(title_text="R_Carré (CV)", row=1, col=2)
 
fig4.update_layout(title="Fig.5 - GridSearchCV : R² en fonction de Alpha",
                   height=400, width=950, template='plotly_white')
fig4.write_image(f"{output_images_path}/f4_gridsearch_alpha.png", scale=2)
fig4.show()

In [17]:
# Fig 5 : Top 10 features (meilleur modèle) 
best_models = {'LinearRegression': lr, f'Ridge (o={ridge_cv.best_params_["alpha"]})': best_ridge,
               f'Lasso (o={lasso_cv.best_params_["alpha"]})': best_lasso}
best_model_obj = list(best_models.values())[best_idx]
 
coef_best = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': best_model_obj.coef_
}).sort_values('Coefficient', key=abs, ascending=True).tail(10)
 
fig5 = go.Figure()
colors_top = ['#e74c3c' if c < 0 else '#27ae60' for c in coef_best['Coefficient']]
fig5.add_trace(go.Bar(
    y=coef_best['Feature'], x=coef_best['Coefficient'].abs(),
    orientation='h', marker_color=colors_top,
    text=[f"{c:+,.0f}" for c in coef_best['Coefficient']],
    textposition='outside', textfont=dict(size=10)
))
fig5.update_layout(title=f"Fig.6 - Top 10 features ({best_model_name})",
                   xaxis_title="Coefficient", yaxis_title="",
                   height=420, width=850, template='plotly_white')
fig5.write_image(f"{output_images_path}/f5_top10_features.png", scale=2)
fig5.show()

In [19]:
# Fig 6 : Effet de la régularisation (norme des coefficients Store)
store_cols = [c for c in X_train.columns if c.startswith('Store_')]
store_coefs_lr = np.abs(lr.coef_[[list(X_train.columns).index(c) for c in store_cols]])
store_coefs_ridge = np.abs(best_ridge.coef_[[list(X_train.columns).index(c) for c in store_cols]])
store_coefs_lasso = np.abs(best_lasso.coef_[[list(X_train.columns).index(c) for c in store_cols]])
 
fig6 = go.Figure()
fig6.add_trace(go.Bar(x=store_cols, y=store_coefs_lr, name='LinearReg',
                       marker_color='#3498db', opacity=0.6))
fig6.add_trace(go.Bar(x=store_cols, y=store_coefs_ridge, name='Ridge',
                       marker_color='#e67e22', opacity=0.7))
fig6.add_trace(go.Bar(x=store_cols, y=store_coefs_lasso, name='Lasso',
                       marker_color='#27ae60', opacity=0.8))
                       
fig6.update_layout(title="Fig.7 - Effet de la régularisation sur les coefficients Store",
                   xaxis_title="Store", yaxis_title="Coefficient",
                   height=450, width=1050, template='plotly_white',
                   barmode='group', xaxis_tickangle=-45)
fig6.write_image(f"{output_images_path}/f6_regularization_effect.png", scale=2)
fig6.show()

### **Sauvegarde du meilleur modele** 

In [20]:
# Sauvegarder le meilleur modèle
joblib.dump(best_model_obj, f"{output_models_path}/best_model.pkl")
print(f"Meilleur modèle sauvegardé : {output_models_path}/best_model.pkl ({best_model_name})")

# Sauvegarder les 3 modèles
joblib.dump(lr, f"{output_models_path}/model_lr.pkl")
joblib.dump(best_ridge, f"{output_models_path}/model_ridge.pkl")
joblib.dump(best_lasso, f"{output_models_path}/model_lasso.pkl")
print(f"3 modèles sauvegardés : model_lr.pkl, model_ridge.pkl, model_lasso.pkl")

# Sauvegarder le tableau comparatif
comparison.to_csv(f"{output_data_path}/metrics_comparison.csv", index=False)
print(f"Métriques sauvegardées : metrics_comparison.csv")

Meilleur modèle sauvegardé : ../output/models/best_model.pkl (LinearRegression)
3 modèles sauvegardés : model_lr.pkl, model_ridge.pkl, model_lasso.pkl
Métriques sauvegardées : metrics_comparison.csv


### **Test/Validation**

In [22]:
# Charger le transformer et le modèle
ct = joblib.load(f"{output_models_path}/column_transformer.pkl")
model = joblib.load(f"{output_models_path}/best_model.pkl")

new_data = pd.DataFrame({
    'Store': [4],
    'Holiday_Flag': [0],
    'Temperature': [55.0],
    'Fuel_Price': [3.45],
    'CPI': [220.5],
    'Unemployment': [7.5],
    'Year': [2012],
    'Month': [6],
    'Day': [15],
    'DayOfWeek': [4]
})

# Transformer + prédire
X_new = ct.transform(new_data)
prediction = model.predict(X_new)
print(f"Ventes prédites : ${prediction[0]:,.2f}")

Ventes prédites : $2,212,346.19
